#Objective

Same preprocessing for column values for Data preparation - 
1. Fixing the Binary values where missing is actually NO


In [1]:
import polars as pl
from pathlib import Path

In [4]:

df = pl.read_parquet(f'./data/interim/1.1-Hip-Reduced.parquet')

In [5]:

# For columns with exactly 2 unique values where one is 9, recode 9 → 0
cols_to_recode = [
    c for c in df.columns
    if set(df[c].drop_nulls().unique().to_list()) == {9, 0}
    or (
        len(df[c].drop_nulls().unique()) == 2
        and 9 in df[c].drop_nulls().unique().to_list()
    )
]

print(f"Columns to recode (2 unique values, one is 9): {cols_to_recode}")

df = df.with_columns([
    pl.when(pl.col(c) == 9).then(0).otherwise(pl.col(c)).alias(c)
    for c in cols_to_recode
])

print("Done. Value 9 replaced with 0 in the above columns.")


Columns to recode (2 unique values, one is 9): ['Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis']
Done. Value 9 replaced with 0 in the above columns.


In [6]:
# Recode 'Pre-Op Q Assisted By': 9 → 0, anything else → 1
df = df.with_columns(
    pl.when(pl.col("Pre-Op Q Assisted By") == 9)
      .then(0)
      .otherwise(1)
      .alias("Pre-Op Q Assisted By")
)

print("'Pre-Op Q Assisted By' recoded: 9 → 0, all other values → 1")
print(df["Pre-Op Q Assisted By"].value_counts())

#Post-Op Q Assisted By 

df = df.with_columns(
    pl.when(pl.col("Post-Op Q Assisted By") == 9)
      .then(0)
      .otherwise(1)
      .alias("Post-Op Q Assisted By")
)

print("'Post-Op Q Assisted By' recoded: 9 → 0, all other values → 1")
print(df["Post-Op Q Assisted By"].value_counts())


'Pre-Op Q Assisted By' recoded: 9 → 0, all other values → 1
shape: (1, 2)
┌──────────────────────┬────────┐
│ Pre-Op Q Assisted By ┆ count  │
│ ---                  ┆ ---    │
│ i32                  ┆ u32    │
╞══════════════════════╪════════╡
│ 1                    ┆ 124844 │
└──────────────────────┴────────┘
'Post-Op Q Assisted By' recoded: 9 → 0, all other values → 1
shape: (2, 2)
┌───────────────────────┬────────┐
│ Post-Op Q Assisted By ┆ count  │
│ ---                   ┆ ---    │
│ i32                   ┆ u32    │
╞═══════════════════════╪════════╡
│ 1                     ┆ 13243  │
│ 0                     ┆ 111601 │
└───────────────────────┴────────┘


In [11]:
# Replace common missing value indicators with null
# Numeric and string columns must be handled separately — is_in() requires matching types
numeric_dtypes = [pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64]
numeric_replace = [9, 999]
string_replace  = ["*", ""]

numeric_exprs = [
    pl.when(pl.col(c).is_in(numeric_replace)).then(None).otherwise(pl.col(c)).alias(c)
    for c in df.columns if df[c].dtype in numeric_dtypes
]
string_exprs = [
    pl.when(pl.col(c).is_in(string_replace)).then(None).otherwise(pl.col(c)).alias(c)
    for c in df.columns if df[c].dtype == pl.Utf8
]

df = df.with_columns(numeric_exprs + string_exprs)
print(df.null_count())

shape: (1, 82)
┌─────────────┬───────────┬────────────┬──────┬───┬────────────┬────────────┬────────────┬─────────┐
│ Provider    ┆ Procedure ┆ Revision   ┆ Year ┆ … ┆ Hip Replac ┆ Hip Replac ┆ Hip Replac ┆ CSVYear │
│ Code        ┆ ---       ┆ Flag       ┆ ---  ┆   ┆ ement      ┆ ement      ┆ ement OHS  ┆ ---     │
│ ---         ┆ u32       ┆ ---        ┆ u32  ┆   ┆ Post-Op Q  ┆ Post-Op Q  ┆ Post-Op Q  ┆ u32     │
│ u32         ┆           ┆ u32        ┆      ┆   ┆ Work       ┆ Scor…      ┆ …          ┆         │
│             ┆           ┆            ┆      ┆   ┆ ---        ┆ ---        ┆ ---        ┆         │
│             ┆           ┆            ┆      ┆   ┆ u32        ┆ u32        ┆ u32        ┆         │
╞═════════════╪═══════════╪════════════╪══════╪═══╪════════════╪════════════╪════════════╪═════════╡
│ 0           ┆ 0         ┆ 0          ┆ 0    ┆ … ┆ 962        ┆ 1627       ┆ 4038       ┆ 0       │
└─────────────┴───────────┴────────────┴──────┴───┴────────────┴────────────

In [12]:

df.write_parquet(f'./data/interim/2.0-Hip-preprocessing.parquet', compression='gzip')